<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/10_Colab_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 - Memory-safe Colab run

Runs the whole pipeline from the `code/business_entity_resolution` package.

## Why this is RAM-friendly

* **Reverse blocking** — only the reference side (S1) is indexed and kept in
  memory; S2/S3 are streamed chunk by chunk. The old design held the full source
  text lookup **and** the source posting lists, which is several GB per source.
* **Buffered features** — training rows accumulate into small numpy buffers,
  never a Python list of per-row lists.
* **Bucketed inference** — scored pairs are spilled to disk in bounded buckets,
  so the outputs are produced one slice at a time instead of holding every pair.
* **Streaming everywhere** — `chunksize` reads, `usecols`, explicit dtypes,
  `gc.collect()` between stages.

## Colab tips

* Use a **High-RAM** runtime for the first full run if available; the default
  runtime should also work with a small `--chunk-size`.
* On a small runtime use `--chunk-size 50000 --buckets 512`.
* Everything is checkpointed on Drive (`work/`), so if a stage fails you can
  re-run just that stage instead of starting over.

Stages: `normalize -> evaluate-blocking -> train -> tune -> predict -> validate`


## Setup: clone the pipeline and install dependencies

In [ ]:
!free -h
!nvidia-smi || true

import os
import subprocess
import sys

REPO_URL = "https://github.com/kartik815/Amazon-ML-Challenge-2026.git"
CODE_ROOT = "/content/amlc"

if not os.path.isdir(os.path.join(CODE_ROOT, "code")):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_ROOT], check=False)

PIPELINE_ROOT = os.path.join(CODE_ROOT, "code", "business_entity_resolution")
os.chdir(PIPELINE_ROOT)
print("pipeline root:", PIPELINE_ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
print("dependencies ready")


## Paths and memory settings

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

# Raw dataset (train/ and test/ live directly here).
DATA_DIR = os.path.join(
    DRIVE_ROOT, "01_Dataset", "6ab10eb3b23ba_student_resource",
    "student_resource", "dataset"
)

# Checkpoints and outputs are written to Drive so a crash never loses work.
WORK_DIR = os.path.join(DRIVE_ROOT, "03_Experiments", "Pipeline_Work")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "05_Outputs", "Submission", "output")

# Small runtime -> use these; larger runtime -> raise them.
CHUNK_SIZE = 50_000
BUCKETS = 512
ADVANCED = True

for path in [DATA_DIR, WORK_DIR, OUTPUT_DIR]:
    print(path)


## Run the full pipeline

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "run_pipeline.py", "all",
    "--data-dir", DATA_DIR,
    "--work-dir", WORK_DIR,
    "--output-dir", OUTPUT_DIR,
    "--chunk-size", str(CHUNK_SIZE),
    "--buckets", str(BUCKETS),
]
if ADVANCED:
    cmd.append("--advanced-blocking")

print(" ".join(cmd))
result = subprocess.run(cmd)
print("exit code:", result.returncode)


## Measure the blocking ceiling (frozen vs advanced)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "run_pipeline.py", "--stage", "evaluate-blocking",
    "--data-dir", DATA_DIR,
    "--work-dir", WORK_DIR,
    "--chunk-size", str(CHUNK_SIZE),
]
if ADVANCED:
    cmd.append("--advanced-blocking")

print(" ".join(cmd))
subprocess.run(cmd)


## Inspect the submission

In [ ]:
import os

for name in ["matching_results.tsv", "candidate_pairs.tsv"]:
    path = os.path.join(OUTPUT_DIR, name)
    print("=" * 60)
    print(name, "exists:", os.path.exists(path))
    if os.path.exists(path):
        with open(path, encoding="utf-8") as handle:
            for _ in range(5):
                line = handle.readline()
                if not line:
                    break
                print(line.rstrip())

print("\nValidate:")
print(
    "python3 utils/validate_submission.py "
    f"--matching {OUTPUT_DIR}/matching_results.tsv "
    f"--candidate {OUTPUT_DIR}/candidate_pairs.tsv "
    "--test-dir " + os.path.join(DATA_DIR, "test")
)
